# get network data(for pandas, huggingface, opanai)

In [1]:
diff_idx = input("INPUT 'visualization_target for difficulty RUN_ID'(e.g., 111110000): ")

print(f'Figure will be drawn based on the run_id_{diff_idx} dataset')

Figure will be drawn based on the run_id_111110000 dataset


In [2]:
target_tag = 'openai-api'

In [4]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
from glob import glob

from lib.annotation.tools.Result_Prep import Result_Prep as Result_Prep_D


import lib.stats.chowtest as st
from lib.utils.file_io import *
from lib.utils.statistics import *
from lib.utils.settings import set_matplotlib
from matplotlib import pyplot as plt
import matplotlib as mpl
from lib.visualization.plot_generator import PlotGen
import lib.visualization.figure_setting as figure_setting

mpl.rcParams.update(figure_setting.fig_setting)
mpl.rcParams['font.family'] = figure_setting.init_font()


from itertools import combinations
mpl.rcParams.update(figure_setting.fig_setting)
import networkx as nx




In [5]:
rp_prep_diff = Result_Prep_D(diff_idx)
diff_df = rp_prep_diff.data_concat()

In [6]:
df = diff_df[['creationdate', 'id', 'result']].drop_duplicates()
ids = (
    (df.groupby('id')['result'].count() == 1)
    .reset_index(name='is_one')
    .query('is_one == True')['id']
)
result_df = df[df['id'].isin(ids)].copy()
result_df['result'] = result_df['result'].apply(lambda x: 'Basic' if x == "<Difficulty Level>0" else \
                                                ( 'Intermediate' if x == "<Difficulty Level>1" else 'Advanced' ))

In [7]:
tmp = load_df('/mnt/hdd/mghan/so_data_availability/result/tag/run_id_100/data', ['cdate' , 'id' , 'tag', 'cnt', 'tot_cnt', 'pct'])
df_tot = pd.merge(result_df, tmp, on = 'id')    

100%|██████████| 153/153 [00:06<00:00, 22.68it/s]


In [8]:
df_tot[(df_tot['tag'] == 'openai-api') & (df_tot['result'] == 'Basic')]

,creationdate,id,result,cdate,tag,cnt,tot_cnt,pct
14055,2021-10-17,69605281,Basic,2021-10-17,openai-api,1,2,0.500000
32868,2023-01-02,74978793,Basic,2023-01-02,openai-api,1,3,0.333333
36450,2023-03-20,75795286,Basic,2023-03-20,openai-api,1,2,0.500000
36764,2023-03-28,75864073,Basic,2023-03-28,openai-api,1,3,0.333333
46372,2023-10-31,77394472,Basic,2023-10-31,openai-api,1,2,0.500000
49092,2023-12-31,77739422,Basic,2023-12-31,openai-api,1,2,0.500000
50376,2024-01-30,77904664,Basic,2024-01-30,openai-api,1,4,0.250000
55715,2024-05-30,78557135,Basic,2024-05-30,openai-api,1,3,0.333333
57497,2024-07-10,78728847,Basic,2024-07-10,openai-api,1,1,1.000000
60063,2024-09-04,78950333,Basic,2024-09-04,openai-api,1,2,0.500000


In [9]:
tag_tmp = tmp[tmp['id'].isin(ids)]
tag_tmp = pd.merge(tag_tmp, result_df[['id', 'result']], on='id', how='inner')    

In [10]:
tag_tmp_bf = tag_tmp[tag_tmp['cdate'] < "2022-11-30"]
tag_tmp_af = tag_tmp[tag_tmp['cdate'] >= "2022-11-30"]

In [11]:
bf_tag_list = list(tag_tmp_bf['tag'])
af_tag_list = list(tag_tmp_af['tag'])

In [12]:
retained_tag_list = list(set(bf_tag_list) & set(af_tag_list))
gone_tag_list = list(set(bf_tag_list) - set(af_tag_list))
new_tag_list = list(set(af_tag_list) - set(bf_tag_list))

In [13]:
def get_network_list(df, tag):
    q_id_list = df.loc[df['tag'] == tag, 'id'].unique()
    
    tag_df = df.loc[df['id'].isin(q_id_list)]
    tag_df = tag_df.groupby('id')['tag'].apply(list).reset_index(name='tag_list')

    result = tag_df.assign(tag_pair=tag_df['tag_list'].apply(lambda x: list(combinations(x, 2)))
                        ).explode('tag_pair')
    result.dropna(subset=['tag_pair'], inplace=True)
    merge_df = pd.merge(df, result, on='id')

    merge_df['Source'] = merge_df['tag_pair'].apply(lambda x : x[0])
    merge_df['Target'] = merge_df['tag_pair'].apply(lambda x : x[1])

    merge_df['weight'] = 1

    # tag_pair_level = 0 : left, 1 : retained, 2 : new  
    tag_pair_bf = set(merge_df.loc[merge_df['cdate'] <= '2022-11-30', 'tag_pair'])
    tag_pair_af = set(merge_df.loc[merge_df['cdate'] > '2022-11-30', 'tag_pair'])

    retained_tag_pair = list(set(tag_pair_bf) & set(tag_pair_af))
    left_tag_pair = list(set(tag_pair_bf) - set(tag_pair_af))
    new_tag_pair = list(set(tag_pair_af) - set(tag_pair_bf))

    merge_df['tag_pair_level'] = np.where(merge_df['tag_pair'].isin(retained_tag_pair), 1
                    , np.where(merge_df['tag_pair'].isin(new_tag_pair), 2, 0))


    edges = (
    merge_df.groupby(['Source', 'Target', 'result', 'tag_pair_level'], as_index=False)
    .agg({'weight': 'sum'})
    .sort_values(by='weight', ascending=False)
    )


    return edges[edges['result']=='Basic'], edges[edges['result']=='Intermediate'], edges[edges['result']=='Advanced']

    



In [14]:
basic_bf, intermediate_bf, advanced_bf = get_network_list(tag_tmp_bf, target_tag)
basic_af, intermediate_af, advanced_af = get_network_list(tag_tmp_af, target_tag)


basic_edge, intermediate_edge, advanced_edge = get_network_list(tag_tmp, target_tag)


In [15]:
basic_af.to_csv(f'tag_basic_{target_tag}_af.csv', index=False)
intermediate_af.to_csv(f'tag_intermediate_{target_tag}_af.csv', index=False)
advanced_af.to_csv(f'tag_advanced_{target_tag}_af.csv', index=False)

In [16]:
basic_nodes = pd.DataFrame(list(set(basic_edge['Source']) | set(basic_edge['Target']) ),columns=['node'])
intermediate_nodes = pd.DataFrame(list(set(intermediate_edge['Source']) | set(intermediate_edge['Target']) ),columns=['node'])
advanced_nodes = pd.DataFrame(list(set(advanced_edge['Source']) | set(advanced_edge['Target']) ),columns=['node'])

In [17]:
basic_nodes['retained_yn'] = np.where(basic_nodes['node'].isin(retained_tag_list), 1, 0)
basic_nodes['left_yn'] = np.where(basic_nodes['node'].isin(gone_tag_list), 1, 0)
basic_nodes['new_yn'] = np.where(basic_nodes['node'].isin(new_tag_list), 1, 0)

intermediate_nodes['retained_yn'] = np.where(intermediate_nodes['node'].isin(retained_tag_list), 1, 0)
intermediate_nodes['left_yn'] = np.where(intermediate_nodes['node'].isin(gone_tag_list), 1, 0)
intermediate_nodes['new_yn'] = np.where (intermediate_nodes['node'].isin(new_tag_list), 1, 0)

advanced_nodes['retained_yn'] = np.where(advanced_nodes['node'].isin(retained_tag_list), 1, 0)  
advanced_nodes['left_yn'] = np.where(advanced_nodes['node'].isin(gone_tag_list), 1, 0)
advanced_nodes['new_yn'] = np.where(advanced_nodes['node'].isin(new_tag_list), 1, 0)

In [18]:
basic_nodes['tag_level'] = np.where(basic_nodes['node'].isin(retained_tag_list), 1
                    , np.where(basic_nodes['node'].isin(new_tag_list), 2, 0))

intermediate_nodes['tag_level'] = np.where(intermediate_nodes['node'].isin(retained_tag_list), 1
                    , np.where(intermediate_nodes['node'].isin(new_tag_list), 2, 0))

advanced_nodes['tag_level'] = np.where(advanced_nodes['node'].isin(retained_tag_list), 1
                    , np.where(advanced_nodes['node'].isin(new_tag_list), 2, 0))

In [19]:
# PALETTE = {
#     'primary': "#C8C7C7",
#     'accent':  '#E07A1F',
#     'event':   '#C0392B',
#     'neutral': '#1f4e79',
# }

PALETTE = {
    'primary': "#5B8CB8",
    'accent':  '#EDB380',
    'event':   '#C0392B',
    'neutral': '#5B8CB8',
}

def hex_to_rgb(hex_color):
    hex_color = hex_color.lstrip('#')
    return tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))

# 노드 카테고리 -> 색 매핑 (노드의 분류 속성에 맞춰 키를 수정하세요)
NODE_COLOR = {
    0: PALETTE['neutral'],
    1: PALETTE['primary'],
    2: PALETTE['accent'],
}

# 엣지 카테고리(0,1,2) -> 색 매핑
EDGE_COLOR = {
    0: PALETTE['neutral'],
    1: PALETTE['primary'],
    2: PALETTE['accent'],
}

In [20]:
def add_size_individual(G, size_min=10, size_max=50):
    degrees = dict(G.degree())
    dmin, dmax = min(degrees.values()), max(degrees.values())
    for n, d in degrees.items():
        norm = (d - dmin) / (dmax - dmin) if dmax > dmin else 0.5
        size = size_min + norm * (size_max - size_min)
        G.nodes[n]['size'] = size
        G.nodes[n].setdefault('viz', {})['size'] = size
    return G

In [21]:
def create_n_save_network(edge, node, diff_level):
    G = nx.from_pandas_edgelist(
        edge,
        source="Source",
        target="Target",
        edge_attr=True,   # 여러 개면 리스트로, 전부 다 넣고 싶으면 edge_attr=True
        create_using=nx.Graph()  # 방향그래프면 nx.DiGraph()
    )

    # 2. 노드 속성 채우기 (set_node_attributes 활용)
    node_attrs = node.set_index("node").to_dict("index")
    nx.set_node_attributes(G, node_attrs)

    # 1. 노드 색 적용 (노드 속성 이름이 예: "category")
    for n, data in G.nodes(data=True):
        category = data.get("tag_level")
        hex_color = NODE_COLOR.get(category, PALETTE['neutral'])
        r, g, b = hex_to_rgb(hex_color)
        G.nodes[n]["viz"] = {"color": {"r": r, "g": g, "b": b}}

    # 2. 엣지 색 적용
    for u, v, d in G.edges(data=True):
        level = d.get("tag_pair_level")
        hex_color = EDGE_COLOR.get(level, PALETTE['neutral'])
        r, g, b = hex_to_rgb(hex_color)
        G[u][v]["viz"] = {"color": {"r": r, "g": g, "b": b}}


    G_sub = G.copy()
    # G_sub.remove_node('openai-api')

    # eigenvector centrality
    ec = nx.eigenvector_centrality(G_sub, max_iter=1000)
    nx.set_node_attributes(G_sub, ec, "eigenvector_centrality")

    # 상위 11개만 남기기
    top_n = sorted(ec, key=ec.get, reverse=True)[:11]
    nodes_to_remove = [n for n in G_sub.nodes() if n not in top_n]
    G_sub.remove_nodes_from(nodes_to_remove)

    # 이름 보존 + 라벨 (11개니까 다 표시)
    names = {n: str(n) for n in G_sub.nodes()}
    nx.set_node_attributes(G_sub, names, "name")
    nx.set_node_attributes(G_sub, names, "label")   # 11개 다 라벨

    G_sub = add_size_individual(G_sub)

    nx.write_gexf(G_sub, f"{diff_level}_{target_tag}_network.gexf")
    return G_sub

In [22]:
create_n_save_network(basic_edge, basic_nodes, "basic")
create_n_save_network(intermediate_edge, intermediate_nodes, "intermediate")
create_n_save_network(advanced_edge, advanced_nodes, "advanced")


calculate the propotion of 'openai-api'

In [23]:
# extract top bottom tags list from target dataset
tag_tmp = tmp[tmp['id'].isin(ids)]
tag_list = list(tag_tmp[tag_tmp['cdate'] < "2022-11-30"].groupby(['tag'])['pct'].sum().reset_index()\
                .sort_values(by = ['pct'], ascending=False)['tag'])
topic_num = int(len(tag_list)*0.2)
top_list = tag_list[:topic_num]
bot_list = tag_list[-topic_num:]

In [24]:
bb = tag_tmp[tag_tmp['cdate'] < '2022-11-30'].groupby(['tag'])['pct'].sum().reset_index(name='bb_pct').sort_values(by='bb_pct', ascending=False)
aa = tag_tmp[tag_tmp['cdate'] >= '2022-11-30'].groupby(['tag'])['pct'].sum().reset_index(name='aa_pct').sort_values(by='aa_pct', ascending=False)

In [25]:
cc = pd.merge(bb, aa, on='tag')
cc['aa_over_bb'] = (cc['aa_pct']- cc['bb_pct'])/cc['bb_pct']

In [26]:
aa = tag_tmp[tag_tmp['cdate'] >= "2022-11-30"]

In [27]:
dd = cc.sort_values(by='aa_over_bb', ascending=False).reset_index(drop=True)

In [28]:
dd['top_yn'] = np.where(dd['tag'].isin(top_list), 1, 0)
dd['bot_yn'] = np.where(dd['tag'].isin(bot_list), 1, 0)

In [32]:
dd[dd['tag'] =='openai-api']

,tag,bb_pct,aa_pct,aa_over_bb,top_yn,bot_yn
7,openai-api,1.083333,25.5,22.538462,0,0


In [34]:
1.083333/dd['bb_pct'].sum() *100

np.float64(0.008426781616646142)

In [35]:
25.5/dd['aa_pct'].sum() *100

np.float64(0.15020542801183973)